In [ ]:
import pandas as pd
from collections import defaultdict
import pickle
from wrappers.random_forrest_forecaster_wrapper import RandomForrestForecaster
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings(
    "ignore",
    message=".*force_all_finite.*",
    category=FutureWarning,
    module="sklearn"
)

In [ ]:
with open('../data/datasets/data_cleaned.pkl', 'rb') as f:
    states_dfs = pickle.load(f)

In [ ]:
states= [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA", 
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD", 
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ", 
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC", 
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY",
]
horizon = 24
target = 'residential_electricity_price'
n_components = 3

In [ ]:
import os
if not os.path.isdir('plots'):
    os.makedirs('plots')

In [ ]:
rff_metrics = {'state': [], 'mape': []}
for state in states:
    df = states_dfs[state]
    model = RandomForrestForecaster(state, n_componets=n_components,horizon=horizon)
    X_train = df.drop(columns=[target]).iloc[:-horizon]
    y_train = df[target].iloc[:-horizon]
    test_data = df[target].iloc[-horizon:]
    model.fit(X_train, y_train)
    preds = model.predict()
    plot_path = os.path.join('plots', f'rff_{state}.jpg')
    model.plot_forecast(test_data=test_data, save_path=plot_path)
    mape = mean_absolute_percentage_error(preds, test_data)
    rff_metrics['state'].append(state)
    rff_metrics['mape'].append(mape)

rff_metrics = pd.DataFrame(rff_metrics)

In [ ]:
rff_metrics.sort_values(by='mape', ascending=False)

In [ ]:
with open('metrics/rff_metrics.pkl', 'wb') as f:
    pickle.dump(rff_metrics, f)
f.close()